In [1]:
from datetime import datetime
from zoneinfo import ZoneInfo
# from pathlib import Path
import pathlib

import subprocess
import os
import getpass

import STX3KO_analyses as stx
import TwoPUtils as tpu
mm = stx.mouse_metadata


In [2]:
with open('.oak_pwd', 'r') as f:
    oak_pwd = f.read().strip()


In [3]:
mice = list(mm.ctrl_sessions.keys())
print(mice)

['4467331.1', '4467331.2', '4467332.1', '4467332.2', '4467333.1', 'mCherry6', 'mCherry7', 'mCherry8', 'mCherry9']


In [4]:
mm.ctrl_sessions.get(mice[0]).get('sessions')

({'date': '29_11_2020',
  'scene': 'YMaze_LNovel',
  'session': 1,
  'scan': 3,
  'novel_arm': -1,
  'ravel_ind': 0},
 {'date': '30_11_2020',
  'scene': 'YMaze_LNovel',
  'session': 1,
  'scan': 7,
  'novel_arm': -1,
  'ravel_ind': 1},
 {'date': '01_12_2020',
  'scene': 'YMaze_LNovel',
  'session': 1,
  'scan': 4,
  'novel_arm': -1,
  'ravel_ind': 2},
 {'date': '02_12_2020',
  'scene': 'YMaze_LNovel',
  'session': 1,
  'scan': 3,
  'novel_arm': -1,
  'ravel_ind': 3},
 {'date': '03_12_2020',
  'scene': 'YMaze_LNovel',
  'session': 1,
  'scan': 8,
  'novel_arm': -1,
  'ravel_ind': 4})

In [5]:
def rsync_cmd(session, mouse):
    remote_user = "mplitt"
    remote_host = "dtn.sherlock.stanford.edu"
    remote_path = "/oak/stanford/groups/giocomo/mplitt/2P_Data/STX3KO/"
    local_tmp_path = pathlib.Path("/mnt/BigDisk/2P_scratch/")
    local_tmp_path = local_tmp_path / mouse / session.get('date') 
    local_tmp_path.mkdir(parents=True, exist_ok=True)
    
    
    
    cmd = [
        "sshpass", "-p", oak_pwd, 
        "rsync", "-rlt", "--progress",
        f"{remote_user}@{remote_host}:{remote_path}{mouse}/{session.get('date')}/{session.get('scene')}",
        str(local_tmp_path)
    ]
    return cmd

def run_rsync(cmd):
    try:
        subprocess.run(cmd, check=True)
        print("Rsync completed successfully.")
    except subprocess.CalledProcessError as e:
        print(f"Rsync failed with error: {e}")
        
# def convert_binary_to_tiff(s2p_path, )

In [6]:
mouse = mice[1]
sessions = mm.ctrl_sessions.get(mouse).get('sessions')


for session in sessions[:1]:
    print(f"--- Starting {session} ---")
    
    if isinstance(session, tuple):
        for _session in session:
            rsync_cmd = rsync_cmd(_session, mouse)
            run_rsync(rsync_cmd)
    else:
        rsync_cmd = rsync_cmd(session, mouse)
        run_rsync(rsync_cmd)
    
    
    
    # 3. YOUR CONVERSION CODE HERE
    # (e.g., Suite2p or NeuroConv commands for this session)
    print(f"Converting {session} to NWB...")
    
    # 4. Conserve disk space: Remove raw data after successful conversion
    # subprocess.run(["rm", "-rf", local_tmp_path], check=True)
    # os.makedirs(local_tmp_path, exist_ok=True)
    
  


--- Starting {'date': '29_11_2020', 'scene': 'YMaze_LNovel', 'session': 1, 'scan': 4, 'novel_arm': 1, 'ravel_ind': 0} ---
receiving incremental file list
Rsync completed successfully.
Converting {'date': '29_11_2020', 'scene': 'YMaze_LNovel', 'session': 1, 'scan': 4, 'novel_arm': 1, 'ravel_ind': 0} to NWB...


In [7]:
import suite2p
import numpy as np

mouse = mice[0]
sessions = mm.ctrl_sessions.get(mouse).get('sessions')
session = sessions[0]

s2p_path = pathlib.Path("/mnt/BigDisk/2P_scratch/")
s2p_path = s2p_path / mouse / session.get('date') / session.get('scene') / \
    f"{session.get('scene')}_{session.get('session'):03}_{session.get('scan'):03}" / "suite2p"

for data_str in ("data", "data_chan2"):
    binary_file = s2p_path / "plane0" / f"{data_str}.bin"
    tiff_file = s2p_path / "plane0" / f"{data_str}.tiff"
    print(tiff_file.is_file())
    if not tiff_file.is_file():
        ops = np.load(s2p_path / "plane0" / "ops.npy", allow_pickle=True).item()
        binfile = suite2p.io.BinaryFile(ops['Ly'], ops['Lx'], str(binary_file))
        binfile.write_tiff(tiff_file)

True
True


In [8]:
ops = np.load(s2p_path / "plane0" / "ops.npy", allow_pickle=True).item()

In [9]:
ops['date_proc']

datetime.datetime(2021, 3, 12, 15, 57, 26, 771677)

In [10]:
import pynwb

from pynwb import NWBHDF5IO, NWBFile
from pynwb.base import Images
from pynwb.image import GrayscaleImage
from pynwb.ophys import (
    Fluorescence,
    ImageSegmentation,
    OpticalChannel,
    RoiResponseSeries,
    TwoPhotonSeries,
)

session_start_time = datetime.now().astimezone()
nwbfile = NWBFile(
        session_description="suite2p_proc",
        identifier=str(ops["data_path"][0]),
        session_start_time=session_start_time,
    )

device = nwbfile.create_device(
        name="Microscope",
        description="Giocomo lab Neurolabware 2P Scope",
        manufacturer="Neurolabware",
    )
optical_channel0 = OpticalChannel(
        name="Green PMT",
        description="an optical channel",
        emission_lambda=500.0,
    )
optical_channel1 = OpticalChannel(
        name="Red PMT",
        description="an optical channel",
        emission_lambda=600.0,
    )

imaging_plane0 = nwbfile.create_imaging_plane(
        name="ImagingPlane_ch0",
        optical_channel=optical_channel0,
        imaging_rate=ops["fs"],
        description="standard",
        device=device,
        excitation_lambda=980.0,
        indicator="GCaMP7f",
        location="CA1",
        grid_spacing=([2.0, 2.0]),
        grid_spacing_unit="microns",
    )

binary_file = s2p_path / "plane0" / f"data.bin"
binfile = suite2p.io.BinaryFile(ops['Ly'], ops['Lx'], str(binary_file))
image_series_ch0 = TwoPhotonSeries(
        name="TwoPhotonSeries_ch0",
        dimension=[ops["Ly"], ops["Lx"]],
        data=binfile.data,
        imaging_plane=imaging_plane0,
        format="external",
        starting_time=0.0,
        rate=ops["fs"] * ops["nplanes"],
        unit="n.a.",
    )
nwbfile.add_acquisition(image_series_ch0)

imaging_plane1 = nwbfile.create_imaging_plane(
        name="ImagingPlane_ch1",
        optical_channel=optical_channel1,
        imaging_rate=ops["fs"],
        description="standard",
        device=device,
        excitation_lambda=980.0,
        indicator="mCherry",
        location="CA1",
        grid_spacing=([2.0, 2.0]),
        grid_spacing_unit="microns",
    )

binary_file1 = s2p_path / "plane0" / f"data_chan2.bin"
binfile1 = suite2p.io.BinaryFile(ops['Ly'], ops['Lx'], str(binary_file1))
image_series_ch1 = TwoPhotonSeries(
        name="TwoPhotonSeries_ch1",
        dimension=[ops["Ly"], ops["Lx"]],
        data=binfile1.data,
        imaging_plane=imaging_plane1,
        format="external",
        starting_time=0.0,
        rate=ops["fs"] * ops["nplanes"],
        unit="n.a.",
    )
nwbfile.add_acquisition(image_series_ch1)

with NWBHDF5IO(s2p_path / "ophys.nwb", "w") as fio:
        fio.write(nwbfile)

# print(nwbfile)

In [17]:
mat

{'frame': array([  133,   133,   133, ..., 34307, 34307, 34310],
       shape=(97861,), dtype=uint16),
 'line': array([135, 330, 492, ..., 238, 399,  50], shape=(97861,), dtype=uint16),
 'event_id': array([1, 1, 1, ..., 1, 1, 1], shape=(97861,), dtype=uint8),
 'resfreq': 7916,
 'postTriggerSamples': 5000,
 'recordsPerBuffer': 512,
 'bytesPerBuffer': 10240000,
 'channels': 1,
 'ballmotion': array([], dtype=uint8),
 'abort_bit': 0,
 'scanbox_version': 2,
 'scanmode': 1,
 'config': {'wavelength': 980,
  'frames': 0,
  'lines': 512,
  'magnification': 3,
  'magnification_list': array(['1.0', '1.2', '1.4', '1.7', '2.0', '2.4', '2.8', '3.4', '4.0',
         '4.8', '5.7', '6.7', '8.0'], dtype='<U3'),
  'pmt0_gain': 0.75,
  'pmt1_gain': 0.59,
  'knobby': {'pos': {'x': -41.51, 'y': 70.34, 'z': 0.78, 'a': 0},
   'schedule': array([[  0,   0,  10,   0,  30],
          [  0,   0,  10,   0,  60],
          [  0,   0,  10,   0,  90],
          [  0,   0,  10,   0, 120],
          [  0,   0,  10,   0

In [ ]:
from neuroconv import NWBConverter
from neuroconv.datainterfaces import Suite2pSegmentationInterface, TiffImagingInterface
from datetime import datetime
from dateutil.tz import tzlocal
import os

# 1. Define the Custom Converter Class
class MultiChannelConverter(NWBConverter):
    data_interface_classes = dict(
        Suite2p=Suite2pSegmentationInterface,
        # ImagingGreen=TiffImagingInterface,
        # ImagingRed=TiffImagingInterface
    )

# 2. Setup Source Data Paths
# Replace these with your actual paths
suite2p_folder = s2p_path   # Folder containing ops.npy, F.npy, etc.
green_tiff_path = suite2p_folder / "plane0" / "data.tiff" # Converted from data.bin
red_tiff_path = suite2p_folder / "plane0" / "data_chan2.tiff" # Converted from data_chan2.bin

source_data = dict(
    Suite2p=dict(folder_path=suite2p_folder),
    # ImagingGreen=dict(file_path=green_tiff_path, 
    #                 sampling_frequency=15.46,
    #                 verbose=True),
    # ImagingRed=dict(file_path=red_tiff_path,
    #                 sampling_frequency=15.46,
    #                 verbose=True)
)

# 3. Initialize the Converter
converter = MultiChannelConverter(source_data=source_data)

# 4. Extract and Modify Metadata
metadata = converter.get_metadata()

# --- CRITICAL: RENAME THE SERIES ---
# By default, both imaging interfaces might try to use the name "TwoPhotonSeries".
# We must manually rename them in the metadata to avoid conflicts.

# NOTE: The location of these names in the metadata dictionary depends on 
# whether neuroconv initialized them as a list or single entries. 
# We explicitly force the names here.

# Find the entry for the Green channel and rename it
# (Logic assumes the order matches data_interface_classes, but explicit assignment is safer)
# imaging_plane_name = metadata["Ophys"]["ImagingPlane"][0]["name"]
# print(imaging_plane_name)
metadata["Ophys"]["TwoPhotonSeries"] = [

    {
        "name": "TwoPhotonSeriesGreen",
        "imaging_plane": 'ImagingPlane',
        "unit": "n.a.",
        "description": "Green channel raw imaging data (GCaMP)",
        "comments": "Converted from suite2p data.bin",
        "dimension": [512, 796],
    },
    {
        "name": "TwoPhotonSeriesRed",
        "imaging_plane": 'ImagingPlane',
        "unit" :"n.a.",
        "description": "Red channel raw imaging data (tdTomato/structural)",
        "comments": "Converted from suite2p data_chan2.bin",
        "dimension": [512, 796],
    }
]

# Ensure the session start time is set (Required by NWB)
metadata["NWBFile"]["session_start_time"] = datetime.now(tzlocal())

# 5. Run the Conversion
nwbfile_path = "output_multichannel.nwb"

converter.run_conversion(
    nwbfile_path=nwbfile_path,
    metadata=metadata,
    overwrite=True
)

print(f"Successfully created {nwbfile_path}")

/home/mplitt/mambaforge/envs/stx3/lib/python3.10/site-packages/roiextractors/extractors/suite2p/suite2psegmentationextractor.py:96: UserWarning: More than one channel is detected! Please specify which channel you wish to load with the `channel_name` argument. To see what channels are available, call `Suite2pSegmentationExtractor.get_available_channels(folder_path=...)`.
  warn(
/home/mplitt/mambaforge/envs/stx3/lib/python3.10/site-packages/neuroconv/tools/roiextractors/roiextractors.py:178: FutureWarning: get_channel_names is deprecated and will be removed in May 2026 or after.
  channel_name_list = imgextractor.get_channel_names() or ["OpticalChannel"]
/home/mplitt/mambaforge/envs/stx3/lib/python3.10/site-packages/neuroconv/datainterfaces/ophys/suite2p/suite2pdatainterface.py:225: FutureWarning: The 'stub_frames' parameter is deprecated and will be removed on or after February 2026. Use 'stub_samples' instead.
  super().add_to_nwbfile(


ValueError: Cannot add <class 'pynwb.ophys.TwoPhotonSeries'> 'TwoPhotonSeriesGreen' at 0x140022424566800 to dict attribute 'acquisition' in <class 'pynwb.file.NWBFile'> 'root'. <class 'pynwb.ophys.TwoPhotonSeries'> 'TwoPhotonSeriesGreen' at 0x140022425825152 already exists in 'acquisition' and has the same name.

In [9]:
metadata["Ophys"]["TwoPhotonSeries"] 


[{'name': 'TwoPhotonSeries',
  'description': 'Imaging data from two-photon excitation microscopy.',
  'unit': 'n.a.',
  'imaging_plane': 'ImagingPlane',
  'dimension': [512, 796]}]

In [36]:
binfile.write_tiff(s2p_path / "plane0" / "data.tiff")

Frame Range: (0, 34310), y_range: (0, 512), x_range(0, 796)
Tiff has been saved to /mnt/BigDisk/2P_scratch/4467331.1/29_11_2020/YMaze_LNovel/YMaze_LNovel_001_003/suite2p/plane0/data.tiff


In [ ]:
from neuroconv import Suite2pSegmentationInterface
from datetime import datetime


mouse = mice[0]
sessions = mm.ctrl_sessions.get(mouse).get('sessions')
session = sessions[0]

s2p_path = pathlib.Path("/mnt/BigDisk/2P_scratch/")
s2p_path = s2p_path / mouse / session.get('date') / session.get('scene') / "suite2p"
s2p_path.mkdir(parents=True, exist_ok=True)

# Initialize interface with your suite2p output folder
interface = Suite2pSegmentationInterface(folder_path=str(s2p_path))

# Get and customize metadata (highly recommended for NWB)
metadata = interface.get_metadata()
metadata["NWBFile"]["session_start_time"] = datetime.now() # Required

# Write to NWB
interface.run_conversion(nwbfile_path=str(s2p_path / "output_file.nwb"), metadata=metadata)


In [3]:
OPHYS_DATA_PATH = Path('/mnt/BigDisk/2P_scratch/4467331.1/29_11_2020/YMaze_LNovel')

In [5]:
path_to_save_nwbfile = OPHYS_DATA_PATH / 'ymaze.nwb'

In [7]:


file_path = OPHYS_DATA_PATH / "YMaze_LNovel_001_003.sbx"
interface = SbxImagingInterface(file_path=file_path, verbose=False)

metadata = interface.get_metadata()
# For data provenance we add the time zone information to the conversion
session_start_time = datetime(2020, 1, 1, 12, 30, 0, tzinfo=ZoneInfo("US/Pacific"))
metadata["NWBFile"].update(session_start_time=session_start_time)
# Add subject information (required for DANDI upload)
metadata["Subject"] = dict(subject_id="subject1", 
                           species="Mus musculus", 
                           sex="M", 
                           age="P30D")

# Choose a path for saving the nwb file and run the conversion
nwbfile_path = f"{path_to_save_nwbfile}"
interface.run_conversion(nwbfile_path=nwbfile_path, metadata=metadata)

NotImplementedError: The SbxImagingExtractor does not currently support multiple color channels or 3-dimensional depth.If you with to request either of these features, please do so by raising an issue at https://github.com/catalystneuro/roiextractors/issues

In [ ]:
import pandas as pd
from pynwb import NWBHDF5IO
from pynwb.behavior import BehavioralTimeSeries, TimeSeries

# 1. Setup - assume 'df' is your pandas dataframe from the sqlite file
# df = pd.read_sql_query("SELECT * FROM behavior_table", sqlite_conn)
timestamp_column = 'timestamps' # Update this to your actual column name

with NWBHDF5IO('existing_data.nwb', mode='a') as io:
    nwbfile = io.read()
    
    # 2. Get or create the behavior module
    if 'behavior' not in nwbfile.processing:
        beh_module = nwbfile.create_processing_module('behavior', 'behavioral data')
    else:
        beh_module = nwbfile.processing['behavior']

    # 3. Create a container to hold the multiple time series
    beh_ts_container = BehavioralTimeSeries(name='sqlite_behavior_data')

    # 4. Iterate through columns and add to container
    for col in df.columns:
        if col == timestamp_column:
            continue
            
        # Create TimeSeries for each behavioral metric (e.g., licks, speed)
        ts = TimeSeries(
            name=col,
            data=df[col].values,
            timestamps=df[timestamp_column].values,
            unit='arbitrary', # Change to specific units if known (e.g., 'm/s')
            description=f"Behavioral metric: {col}"
        )
        beh_ts_container.add_time_series(ts)

    # 5. Add container to the module and save
    beh_module.add(beh_ts_container)
    io.write(nwbfile)

In [8]:
from datetime import datetime
from zoneinfo import ZoneInfo
from pathlib import Path
from neuroconv import ConverterPipe
from neuroconv.datainterfaces import TiffImagingInterface, Suite2pSegmentationInterface


folder_path= OPHYS_DATA_PATH / "YMaze_LNovel_001_003" / "suite2p"
interface_suite2p = Suite2pSegmentationInterface(folder_path=folder_path, verbose=False)

# Now that we have defined the two interfaces we pass them to the ConverterPipe which will coordinate the
# concurrent conversion of the data
metadata = interface_suite2p.get_metadata()

# For data provenance we add the time zone information to the conversion
session_start_time = datetime(2020, 1, 1, 12, 30, 0, tzinfo=ZoneInfo("US/Pacific"))
metadata["NWBFile"].update(session_start_time=session_start_time)
# Add subject information (required for DANDI upload)
metadata["Subject"] = dict(subject_id="subject1", species="Mus musculus", sex="M", age="P30D")

# Choose a path for saving the nwb file and run the conversion
nwbfile_path = f"{path_to_save_nwbfile}"
interface_suite2p.run_conversion(nwbfile_path=nwbfile_path,  metadata=metadata)

/home/mplitt/mambaforge/envs/stx3/lib/python3.10/site-packages/roiextractors/extractors/suite2p/suite2psegmentationextractor.py:96: UserWarning: More than one channel is detected! Please specify which channel you wish to load with the `channel_name` argument. To see what channels are available, call `Suite2pSegmentationExtractor.get_available_channels(folder_path=...)`.
  warn(
/home/mplitt/mambaforge/envs/stx3/lib/python3.10/site-packages/neuroconv/datainterfaces/ophys/suite2p/suite2pdatainterface.py:225: FutureWarning: The 'stub_frames' parameter is deprecated and will be removed on or after February 2026. Use 'stub_samples' instead.
  super().add_to_nwbfile(


In [9]:
metadata

DeepDict(
{'NWBFile': {'session_description': '',
  'identifier': '1e3eed78-9225-4fc0-a95c-4701f1f39a75',
  'source_script': 'Created using NeuroConv v0.9.1',
  'source_script_file_name': '/home/mplitt/mambaforge/envs/stx3/lib/python3.10/site-packages/neuroconv/basedatainterface.py',
  'session_start_time': datetime.datetime(2020, 1, 1, 12, 30, tzinfo=zoneinfo.ZoneInfo(key='US/Pacific'))},
 'Ophys': {'Device': [{'name': 'Microscope'}],
  'ImagingPlane': [{'name': 'ImagingPlaneChan1Plane0',
    'description': 'The plane or volume being imaged by the microscope.',
    'excitation_lambda': nan,
    'indicator': 'unknown',
    'location': 'unknown',
    'device': 'Microscope',
    'optical_channel': [{'name': 'OpticalChannel',
      'emission_lambda': nan,
      'description': 'An optical channel of the microscope.'}]}],
  'Fluorescence': {'name': 'Fluorescence',
   'BackgroundPlaneSegmentation': {'neuropil': {'name': 'neuropil',
     'description': 'Array of neuropil traces.',
     'unit'

In [11]:
import suite2p
from suite2p.io import nwb

In [12]:
nwb.save_nwb(OPHYS_DATA_PATH / "YMaze_LNovel_001_003" / "suite2p")

root pynwb.file.NWBFile at 0x140386183063648
Fields:
  file_create_date: [datetime.datetime(2026, 2, 3, 21, 12, 51, 551700, tzinfo=tzlocal())]
  identifier: /media/mplitt/BigDisk/2P_scratch/4467331.1/29_11_2020/YMaze_LNovel/YMaze_LNovel_001_003
  session_description: suite2p_proc
  session_start_time: 2021-03-12 15:57:26.771677-08:00
  timestamps_reference_time: 2021-03-12 15:57:26.771677-08:00



/home/mplitt/mambaforge/envs/stx3/lib/python3.10/site-packages/hdmf/container.py:542: UserWarning: The linked table for DynamicTableRegion 'rois' does not share an ancestor with the DynamicTableRegion.
  child._validate_on_set_parent()
/home/mplitt/mambaforge/envs/stx3/lib/python3.10/site-packages/suite2p/io/nwb.py:494: FutureWarning: Images.__init__: Using positional arguments for this method is discouraged and will be deprecated in a future major release. Please use keyword arguments to ensure future compatibility.
  images = Images("Backgrounds_%d" % iplane)


In [13]:
import pynwb